# Семинар 1. Первый установочный
#
Сегодня свяжем понятия из лекции с кодом: таблица → признаки и ответ → обучение → прогноз → ошибка.
Сначала разберём примеры, затем построим первый эксперимент на обращениях в поддержку.
#
К концу занятия лучше всего получить заполненный и выполненный jupyter.

## Переходим к коду

`маркер=START`

Предсказываем **время решения обращения в момент его регистрации**. В исторических данных
время решения уже известно. У нового, разумеется, ещё нет..

| Теория | Jupyter |
|---|---|
| Объект | Одна строка таблицы: одно обращение |
| Признаки | Выбранные столбцы, доступные в момент регистрации |
| Целевая переменная | `resolution_minutes`: время до решения в минутах |
| Обучение | `fit(X_train, y_train)` |
| Предсказание | `predict(X_valid)` |
| Проверка | Сравнение прогнозов с `y_valid` по одной метрике |

**Подумать.** Представьте, что в историческом архиве есть полное число сообщений до закрытия обращения.
Можно ли его передать модели, которая должна ответить при регистрации?

## Что такое Jupyter Notebook?

`маркер=ENV`

Notebook состоит из текстовых и кодовых ячеек. Код выполняет выбранный Python-процесс -- **kernel**.
Переменные сохраняются между выполнениями. Изменить текст ячейки недостаточно: её надо снова выполнить.
После перезапуска kernel переменные создаются заново, поэтому рабочий порядок -- сверху вниз.

В Jupyter кодовую ячейку обычно запускают `Shift+Enter`; в редакторе можно использовать кнопку выполнения.
Пока запускайте ячейки последовательно, а не весь notebook сразу: впереди есть вопросы до запуска.

In [1]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Исполняемый файл:", sys.executable)
print("Рабочая папка:", Path.cwd())

example_minutes = 12
print("Пример вычисления:", example_minutes + 3)

Python: 3.14.3
Исполняемый файл: c:\msys64\ucrt64\bin\python.exe
Рабочая папка: C:\Users\АРТЕМ\Desktop\Mephi\MlLabs(man)\sem1
Пример вычисления: 15


Теперь библиотеки. NumPy нужен для массивов и вычислений, pandas для таблиц,
scikit-learn для моделей и метрик. `display` выводит таблицу в читаемом виде.

In [2]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

NumPy: 2.4.1
pandas: 2.3.3
scikit-learn: 1.8.0


## Таблица, один столбец и матрица признаков

`маркер=TABLE` · **Вместе**

Ниже три вымышленных обращения, составленных вручную только для объяснения кода.
Это **не замена** основному CSV и не данные для оценки качества модели.

`DataFrame` -- таблица с именованными столбцами. Индекс строк слева -- не обязательно признак и не обязательно ID обращения.

In [3]:
tiny = pd.DataFrame(
    {
        "ticket_id": ["demo-1", "demo-2", "demo-3"],
        "message_length": [40, 120, 250],
        "queue_at_open": [0, 2, 5],
        "account_age_days": [10, 365, 90],
        "resolution_minutes": [15.0, 50.0, 100.0],
    }
)
display(tiny)

,ticket_id,message_length,queue_at_open,account_age_days,resolution_minutes
0,demo-1,40,0,10,15.0
1,demo-2,120,2,365,50.0
2,demo-3,250,5,90,100.0


В нашем эксперименте заранее выбраны три входа и один ответ:

In [4]:
FEATURES = ["message_length", "queue_at_open", "account_age_days"]
TARGET = "resolution_minutes"

tiny_X = tiny[FEATURES]  # Таблица выбранных входных столбцов.
tiny_y = tiny[TARGET]  # Один столбец с правильными ответами.
display(tiny_X)
display(tiny_y)

,message_length,queue_at_open,account_age_days
0,40,0,10
1,120,2,365
2,250,5,90


0     15.0
1     50.0
2    100.0
Name: resolution_minutes, dtype: float64

**Форма массивов.**

`array.shape` -> ? Форма объекта

In [5]:
print("X:", type(tiny_X).__name__, tiny_X.shape)
print("Один столбец:", type(tiny_y).__name__, tiny_y.shape)
print("Таблица из одного столбца:", type(tiny[[TARGET]]).__name__, tiny[[TARGET]].shape)

X: DataFrame (3, 3)
Один столбец: Series (3,)
Таблица из одного столбца: DataFrame (3, 1)


В следующей ячейке выведите только `queue_at_open` сначала как столбец, затем как таблицу из одного столбца.
Посмотрите на `.shape`.

In [6]:
queue_series = tiny["queue_at_open"]
queue_frame = tiny[["queue_at_open"]]
print(type(queue_series).__name__, queue_series.shape)
print(type(queue_frame).__name__, queue_frame.shape)

Series (3,)
DataFrame (3, 1)


## Переходим к формулам

`маркер=MAE`

В NumPy арифметика с массивом часто выполняется поэлементно. В отличие от списка Python,
массив позволяет, например, прибавить число к каждому элементу.

In [7]:
values_list = [10, 20, 30]
values_array = np.array(values_list, dtype=float)

print("Списки соединяются:", values_list + [3])
print("К элементам массива прибавляется 3:", values_array + 3)
print("Среднее массива:", np.mean(values_array))

Списки соединяются: [10, 20, 30, 3]
К элементам массива прибавляется 3: [13. 23. 33.]
Среднее массива: 20.0


Для правильных ответов $y_i$ и прогнозов $\hat y_i$ средняя абсолютная ошибка:

$$\operatorname{MAE}=\frac{1}{n}\sum_{i=1}^n |y_i-\hat y_i|.$$

Здесь $n$ -- число проверяемых объектов. Чем меньше MAE, тем лучше по этой метрике.

In [8]:
toy_y = np.array([1.0, 3.0, 5.0])
toy_prediction = np.array([2.0, 3.0, 2.0])

mae_by_hand = (abs(1.0 - 2.0) + abs(3.0 - 3.0) + abs(5.0 - 2.0)) / 3
mae_by_numpy = np.mean(np.abs(toy_y - toy_prediction))

if mae_by_hand is None or mae_by_numpy is None:
    print("Попытка пока не завершена: заполните два значения и объяснение.")
else:
    print("Вручную:", mae_by_hand, "| NumPy:", mae_by_numpy)
    print("Два ваших вычисления совпали:", np.isclose(mae_by_hand, mae_by_numpy))

Вручную: 1.3333333333333333 | NumPy: 1.3333333333333333
Два ваших вычисления совпали: True


## Загружаем данные и описываем задачу

`маркер=E1`

Переходим к набору `intro_tickets.csv` из курса.
Ссылка https://gokapi.od.likemath.me:18443/d?id=dPlZsax5ijeUQwpI
Путь ниже сохранён относительно корня проекта; личные абсолютные пути не нужны.
Вспомогательная функция ищет корень проекта вверх от рабочей папки.

In [9]:
DATA_RELATIVE = Path(
    "materials/intro_tickets.csv"
)


def find_course_root():
    starts = [Path.cwd()]
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / DATA_RELATIVE).is_file():
                return candidate
    raise FileNotFoundError(
        f"Не найден {DATA_RELATIVE}. Рабочая папка: {Path.cwd()}. "
        "Откройте notebook внутри исходного проекта курса с папкой materials. "
        "CSV берётся из курса, автоматически новый набор не создаётся."
    )


ROOT = find_course_root()
df = pd.read_csv(ROOT / DATA_RELATIVE)
print("Файл:", ROOT / DATA_RELATIVE)

required_columns = ["ticket_id", *FEATURES, TARGET]
missing_columns = set(required_columns) - set(df.columns)
assert not missing_columns, f"В файле не хватает столбцов: {missing_columns}"
assert len(df) >= 8, (
    "Для этого упражнения нужен основной набор, а не таблица из трёх строк."
)
assert df["ticket_id"].notna().all() and df["ticket_id"].is_unique
assert np.isfinite(df[FEATURES + [TARGET]].to_numpy(dtype=float)).all(), (
    "В числовой части вводного набора есть пропуски или бесконечности. "
    "Проверьте, что открыт правильный CSV; не заполняйте их молча."
)

print("Размер таблицы:", df.shape)
print("Пропуски по столбцам:\n", df.isna().sum())
print("Повторяющиеся строки:", df.duplicated().sum())
display(df.head())
display(min(df["queue_at_open"]))
display(max(df["queue_at_open"]))

Файл: C:\Users\АРТЕМ\Desktop\Mephi\MlLabs(man)\sem1\materials\intro_tickets.csv
Размер таблицы: (480, 5)
Пропуски по столбцам:
 ticket_id             0
message_length        0
queue_at_open         0
account_age_days      0
resolution_minutes    0
dtype: int64
Повторяющиеся строки: 0


,ticket_id,message_length,queue_at_open,account_age_days,resolution_minutes
0,I0000,1160,23,422,159.21
1,I0001,1009,19,670,134.78
2,I0002,1501,6,535,137.42
3,I0003,1210,21,387,141.41
4,I0004,670,2,944,58.32


0

29

*Составьте паспорт данных и три проверяемых наблюдения.
Для одного числового поля укажите диапазон; при необходимости добавьте команду с `.min()` и `.max()`.
Достаточно коротких формулировок, но не просто «таблица загрузилась».

### Что мы знаем о данных?

- Одна строка означает: …
- Что предсказываем и в каких единицах: …
- В какой момент предсказываем: …
- Три допустимых входных поля: …
- Можно ли брать `ticket_id` и сам target в признаки? …

### Что удалось получить?

1. Размер таблицы: …
2. Что проверено о пропусках и повторах: …
3. Диапазон выбранного числового поля: …

In [10]:
for col in FEATURES + [TARGET]:
    print(col, float(df[col].min()), float(df[col].max()))
assert df["queue_at_open"].ge(0).all()

message_length 47.0 1798.0
queue_at_open 0.0 29.0
account_age_days 1.0 1455.0
resolution_minutes 22.93 200.29


## Отделяем обучение от проверки

`маркер=E2-SPLIT`

На **train** модель получает пары «признаки -- правильный ответ» и обучается.
На **validation** сначала получает только признаки и делает прогнозы, после чего мы сравниваем их с известными ответами.
`y_valid` читать для вычисления метрики можно; обучать на нём рассматриваемую модель -- нет.

Используем фиксированное случайное разбиение 75% / 25%.

In [11]:
from sklearn.model_selection import train_test_split

train, valid = train_test_split(df, test_size=0.25, random_state=17)
assert set(train["ticket_id"]).isdisjoint(set(valid["ticket_id"]))

X_train, y_train = train[FEATURES], train[TARGET]
X_valid, y_valid = valid[FEATURES], valid[TARGET]
assert X_train.index.equals(y_train.index)
assert X_valid.index.equals(y_valid.index)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)

Train: (360, 3) (360,)
Validation: (120, 3) (120,)


**Для самопроверки.**

1. Почему опасно перемешать `X` и `y` независимо друг от друга?
2. Что именно фиксирует `random_state=17`? Почему не меняем его, пока результат не станет красивее?


## Строим опорное решение (baseline) без использования признаков

`маркер=E2-BASELINE`

Baseline -- простое опорное решение, с которым сравниваем следующий подход.
Сегодня оно предсказывает одно число: медиану правильных ответов **обучающей** части.
Для MAE медиана -- естественный выбор константы.

**Перед началом:** из какой части данных берём медиану и почему?
Сколько чисел вернёт такое правило на validation? Будут ли они различаться для двух обращений с разными признаками?

In [12]:
median_minutes = float(y_train.median())
manual_baseline_prediction = np.full(len(X_valid), median_minutes)
manual_baseline_mae = np.mean(np.abs(y_valid.to_numpy() - manual_baseline_prediction))

print("Константа, мин:", median_minutes)
print("Первые прогнозы:", manual_baseline_prediction[:5])
print("MAE NumPy-baseline, мин:", manual_baseline_mae)

Константа, мин: 110.535
Первые прогнозы: [110.535 110.535 110.535 110.535 110.535]
MAE NumPy-baseline, мин: 33.0755


Та же идея реализована в `DummyRegressor`. У него такой же интерфейс `fit`/`predict`, как у более сложных моделей.
Интерфейсы сейчас out-of-scope.

In [13]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_valid)
baseline_mae = mean_absolute_error(y_valid, pred_base)

assert pred_base.shape == y_valid.shape
assert np.allclose(pred_base, manual_baseline_prediction)
assert np.isclose(baseline_mae, manual_baseline_mae)
print("MAE sklearn-baseline, мин:", baseline_mae)

MAE sklearn-baseline, мин: 33.0755


## Обучаем модель и смотрим результат

`маркер=E3`

Сравним baseline с готовой цепочкой `StandardScaler → Ridge`.
Scaler приводит числовые признаки к сопоставимому масштабу, а Ridge -- обучаемая линейная модель.
Математическое устройство модели и подробности предобработки будут разбираться на следующих занятиях.
Сейчас важно понимать, что передаётся на вход и где происходит обучение.

В `model.fit(X_train, y_train)` обучаются преобразование и модель. В `model.predict(X_valid)`
они применяются без нового обучения. MAE ниже -- наша метрика сравнения;

**До запуска.** Какой формы ожидаете `y_hat`? Нужен ли `y_valid` для вызова `predict`?
А для вычисления MAE? Почему эти два вопроса имеют разные ответы?

In [14]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model.fit(X_train, y_train)
y_hat = model.predict(X_valid)

assert y_hat.shape == y_valid.shape
assert np.isfinite(y_hat).all()
model_mae = mean_absolute_error(y_valid, y_hat)
model_mae_numpy = np.mean(np.abs(y_valid.to_numpy() - y_hat))
assert np.isclose(model_mae, model_mae_numpy)

print("Формы ответа и прогноза:", y_valid.shape, y_hat.shape)
display(
    pd.DataFrame(
        {
            "вариант": ["Медианный baseline", "Ridge: все три признака"],
            "MAE, мин": [baseline_mae, model_mae],
        }
    )
)

Формы ответа и прогноза: (120,) (120,)


,вариант,"MAE, мин"
0,Медианный baseline,33.075500
1,Ridge: все три признака,6.732782


**Обсуждение.** Какая строка таблицы лучше по MAE на текущем разбиении? Насколько отличается ошибка в минутах?
Разница в средних ошибках не означает, что одна модель лучше предсказывает каждое отдельное обращение.

In [15]:
comparison = pd.DataFrame(
    {
        "ticket_id": valid["ticket_id"].to_numpy(),
        "правильный ответ": y_valid.to_numpy(),
        "baseline": pred_base,
        "Ridge": y_hat,
        "абсолютная ошибка Ridge": np.abs(y_valid.to_numpy() - y_hat),
    }
)
display(comparison.head())

,ticket_id,правильный ответ,baseline,Ridge,абсолютная ошибка Ridge
0,I0122,132.48,110.535,129.082587,3.397413
1,I0029,128.97,110.535,113.950994,15.019006
2,I0301,123.25,110.535,111.387129,11.862871
3,I0173,86.04,110.535,93.546697,7.506697
4,I0226,102.32,110.535,104.100756,1.780756


## Проверяем собственную гипотезу

`маркер=E4`

Выберите один из трёх признаков и проверьте, что изменится, если его убрать.
Например, можно исследовать роль `queue_at_open`, но разрешён другой выбор из `FEATURES`.
До запуска запишите ожидаемое направление изменения MAE и объяснение.
Неверный прогноз результата -- не неудача: важен корректный эксперимент и вывод после него.

Меняем **только набор признаков**. Данные и их разбиение, ответы, модель, `alpha=1.0` и метрика остаются прежними.
Создаём и обучаем новую цепочку: у неё теперь другой набор входов.
Не меняйте seed и не используйте ID или target вместо удалённого признака.

In [16]:
hypothesis = "account_age_days не сильно поменяет запрос"
candidate_features = [f for f in FEATURES if f != "account_age_days"] 

Далее код обучения дан. Прочитайте его, затем запустите после заполнения предыдущей ячейки.
Неизменённый список, его перестановка или пустая гипотеза не считаются выполненной попыткой.

In [17]:
candidate = None
candidate_prediction = None
candidate_mae = None

candidate_ready = (
    isinstance(hypothesis, str)
    and bool(hypothesis.strip())
    and isinstance(candidate_features, list)
    and all(isinstance(name, str) and name in FEATURES for name in candidate_features)
    and len(candidate_features) == len(FEATURES) - 1
    and len(set(candidate_features)) == len(candidate_features)
)

if not candidate_ready:
    print(
        "Эксперимент ещё не выполнен: запишите гипотезу и уберите ровно один из трёх признаков."
    )
else:
    candidate = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    candidate.fit(train[candidate_features], y_train)
    candidate_prediction = candidate.predict(valid[candidate_features])
    assert candidate_prediction.shape == y_valid.shape
    assert np.isfinite(candidate_prediction).all()
    candidate_mae = mean_absolute_error(y_valid, candidate_prediction)
    print("Гипотеза:", hypothesis)
    print("Признаки:", candidate_features)
    print(
        "Изменение MAE относительно модели со всеми признаками, мин:",
        candidate_mae - model_mae,
    )

Гипотеза: account_age_days не сильно поменяет запрос
Признаки: ['message_length', 'queue_at_open']
Изменение MAE относительно модели со всеми признаками, мин: 0.15784695531572712


In [18]:
result_rows = [
    {"вариант": "Медианный baseline", "MAE, мин": baseline_mae},
    {"вариант": "Ridge: все три признака", "MAE, мин": model_mae},
]
if candidate_mae is not None:
    result_rows.append({"вариант": "Ridge: мой эксперимент", "MAE, мин": candidate_mae})
results = pd.DataFrame(result_rows)
display(results)

,вариант,"MAE, мин"
0,Медианный baseline,33.075500
1,Ridge: все три признака,6.732782
2,Ridge: мой эксперимент,6.890628


### Разбор

- Какой фактор изменили: …
- Что оставили неизменным: …
- Как изменилась MAE в минутах: …
- Совпало ли это с гипотезой: …
- Что этот эксперимент доказывает/не доказывает: …

Программа проверяет наличие непустой записи и допустимого списка, а не содержательность гипотезы.
Два заполненных поля не заменяют объяснение результата.

## Фиксируем результат

`маркер=E5`

Проверьте, что notebook сохранён, **Restart Kernel → Run All** ничего не ломает и восстанавливает ответы